# Controlled T4 offline and streaming benchmark

Use a fresh Colab GPU runtime. The gateway, vLLM server and client run on localhost. This is not a cloud deployment or a production capacity test. The runtime installs pinned vLLM from PyPI and downloads the public model and gateway source.


In [ ]:
%pip install -q vllm==0.33.0


In [ ]:
import torch, vllm
MODEL="Qwen/Qwen2.5-0.5B-Instruct"
import gc, json, time, random, statistics, pathlib
from vllm import LLM, SamplingParams
REV="7ae557604adf67be50417f59c2c2f167def9a775"
llm2=LLM(model=MODEL, revision=REV, dtype="half", max_model_len=1024, gpu_memory_utilization=0.80, enforce_eager=True, enable_prefix_caching=False)
print("PREFIX_CACHE_ENABLED", llm2.llm_engine.vllm_config.cache_config.enable_prefix_caching, flush=True)
params2=SamplingParams(temperature=0, max_tokens=64, ignore_eos=True)
fixed_prompts=[f"Explain a practical way to monitor model serving latency. Example {i}:" for i in range(8)]
for n in [1,4,8]:
    llm2.generate(fixed_prompts[:n], params2, use_tqdm=False)
rng=random.Random(20261115)
offline=[]
for repeat in range(7):
    order=[1,4,8]
    rng.shuffle(order)
    for n in order:
        start=time.perf_counter()
        out=llm2.generate(fixed_prompts[:n], params2, use_tqdm=False)
        secs=time.perf_counter()-start
        count=sum(len(o.outputs[0].token_ids) for o in out)
        row={"repeat":repeat+1,"batch_size":n,"seconds":secs,"output_tokens":count,"output_tokens_per_s":count/secs}
        offline.append(row)
        print("TRIAL",json.dumps(row),flush=True)
print("OFFLINE_SUMMARY",json.dumps({str(n):{"median":statistics.median(x["output_tokens_per_s"] for x in offline if x["batch_size"]==n),"min":min(x["output_tokens_per_s"] for x in offline if x["batch_size"]==n),"max":max(x["output_tokens_per_s"] for x in offline if x["batch_size"]==n)} for n in [1,4,8]}),flush=True)

In [ ]:
import gc, subprocess, sys, urllib.request, pathlib, time
llm2.llm_engine.engine_core.shutdown()
del llm2
gc.collect()
torch.cuda.empty_cache()
GATEWAY_SHA="b57e068fff4be667d7e84bb2199fec288db81cbb"
url=f"https://raw.githubusercontent.com/nikhil-s-dev/nikhil-s-dev.github.io/{GATEWAY_SHA}/projects/inference-gateway/gateway.py"
pathlib.Path("gateway.py").write_bytes(urllib.request.urlopen(url, timeout=30).read())
logfile=open("/tmp/vllm-serving.log","w")
cmd=[sys.executable,"-m","vllm.entrypoints.openai.api_server","--model",MODEL,"--revision",REV,"--dtype","half","--max-model-len","1024","--gpu-memory-utilization","0.80","--enforce-eager","--disable-prefix-caching","--host","127.0.0.1","--port","8000"]
server=subprocess.Popen(cmd,stdout=logfile,stderr=subprocess.STDOUT)
ready=False
for _ in range(120):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health",timeout=2) as r:
            ready=r.status==200
        if ready: break
    except Exception: time.sleep(2)
if not ready:
    print(pathlib.Path("/tmp/vllm-serving.log").read_text()[-7000:])
    raise RuntimeError("vLLM server did not become ready")
gateway=subprocess.Popen([sys.executable,"gateway.py","--upstream","http://127.0.0.1:8000","--port","18080","--max-inflight","8"],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
assert urllib.request.urlopen("http://127.0.0.1:18080/health",timeout=3).status==200
print("READY: vLLM 127.0.0.1:8000; gateway 127.0.0.1:18080; cache disabled; model revision pinned; gateway commit",GATEWAY_SHA,flush=True)

In [ ]:
import http.client, json, time, statistics, math, random, concurrent.futures
PAYLOAD=json.dumps({"model":MODEL,"messages":[{"role":"user","content":"Explain three ways to monitor model serving latency."}],"stream":True,"stream_options":{"include_usage":True},"temperature":0,"max_tokens":32,"ignore_eos":True}).encode()
def request_one(port):
    t0=time.perf_counter()
    conn=http.client.HTTPConnection("127.0.0.1",port,timeout=90)
    ttft=None; done=False; tokens=None; status=None
    try:
        conn.request("POST","/v1/chat/completions",body=PAYLOAD,headers={"Content-Type":"application/json","Accept":"text/event-stream"})
        r=conn.getresponse(); status=r.status
        if status != 200:
            r.read()
            return {"ok":False,"status":status,"error":"http", "e2e_s":time.perf_counter()-t0}
        while True:
            line=r.readline()
            if not line: break
            line=line.strip()
            if not line.startswith(b"data:"): continue
            data=line[5:].strip()
            if data == b"[DONE]":
                done=True
                break
            chunk=json.loads(data)
            choices=chunk.get("choices") or []
            if ttft is None and choices and (choices[0].get("delta") or {}).get("content"):
                ttft=time.perf_counter()-t0
            if chunk.get("usage"):
                tokens=chunk["usage"].get("completion_tokens")
        return {"ok":bool(done and ttft is not None and tokens is not None),"status":status,"ttft_s":ttft,"e2e_s":time.perf_counter()-t0,"tokens":tokens,"done":done}
    except Exception as e:
        return {"ok":False,"status":status,"error":type(e).__name__,"e2e_s":time.perf_counter()-t0}
    finally:
        conn.close()
print("DIRECT_SMOKE",request_one(8000),flush=True)
print("GATEWAY_SMOKE",request_one(18080),flush=True)

In [ ]:
def batch_requests(port, concurrency, count):
    start=time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
        reqs=list(pool.map(request_one,[port]*count))
    return time.perf_counter()-start, reqs
def p95(values):
    v=sorted(values)
    return v[math.ceil(.95*len(v))-1] if v else None
for label,port in [("direct",8000),("gateway",18080)]:
    _,warm=batch_requests(port,8,8)
    assert all(x["ok"] for x in warm), (label,warm)
    print("WARMUP",label,len(warm),"success",flush=True)
server_runs=[]
rng2=random.Random(20261115)
for rep in range(1,4):
    groups=[(name,port,c) for name,port in [("direct",8000),("gateway",18080)] for c in [1,4,8]]
    rng2.shuffle(groups)
    for name,port,c in groups:
        elapsed, reqs=batch_requests(port,c,12)
        ok=[x for x in reqs if x["ok"]]
        tt=[x["ttft_s"] for x in ok]
        run={"repeat":rep,"path":name,"concurrency":c,"requests":12,"elapsed_s":elapsed,"errors":12-len(ok),"ttft_p50_s":statistics.median(tt) if tt else None,"ttft_p95_s":p95(tt),"output_tokens_per_s":sum(x["tokens"] for x in ok)/elapsed,"completed_per_s":len(ok)/elapsed,"samples":reqs}
        server_runs.append(run)
        print("SERVING_TRIAL",json.dumps({k:v for k,v in run.items() if k!="samples"}),flush=True)
print("SERVING_FINISHED",len(server_runs),"groups",flush=True)

In [ ]:
import json
with open("controlled-results.json","w") as f: json.dump({"offline_trials":offline,"serving_trials":server_runs}, f, indent=2)
print("Saved controlled-results.json")


In [ ]:
for proc in (gateway, server):
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        print("Process still stopping:", proc.pid)
print("Server processes stopped")
